# Readouts: is the edge-choosing layer interpretable?

`g-both-angle` chooses, for each token in each layer, a weighting over 6 edges (1, 2, 4, 16, 64 and 127 tokens back). It can also use sin(angle) between the two tokens: 0 = same direction, 1 = completely different. This notebook trains it (2 seeds × 5 epochs, saving the weights) and runs three tests:

1. **Readable:** which edges each layer uses, and how peaked the choice is. Are the patterns the same across seeds?
2. **Used:** remove one edge at a time and measure how much worse the model gets. Edges that look important should matter when removed.
3. **Meaningful:** is sin(angle) between a token and the previous one higher right after a sentence ends?

About 15 minutes on an A100 (estimate). Validation only.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/readouts.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
!python -m grassman.data --data-dir {DATA}

In [ ]:
# Run every model of this notebook on the same GPU type: precision differs (bf16 on A100/L4, fp16 on T4).
# COMPILE = True uses torch.compile for every model (fair across models); turn it on only if speed.ipynb
# showed compiled runs are faster on this GPU, since compiling costs time per run.
COMPILE = False

def train(mixer, seed, epochs=5, lr=1e-3, attn='', save=None, tag='pilot'):
    cmd = [sys.executable, '-m', 'grassman.train', '--mixer', mixer, '--seed', str(seed), '--lr', str(lr),
           '--epochs', str(epochs), '--tag', tag, '--data-dir', DATA, '--out', RUNS,
           '--skip-existing', '--no-test']
    if attn:
        cmd += ['--attn-layers', attn]
    if save:
        cmd += ['--save', save]
    if COMPILE:
        cmd += ['--compile']
    print('>>', mixer, f'attn@{attn}' if attn else '', 'seed', seed, flush=True)
    subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
SEEDS = [0, 1]
CKPT = {s: f'{OUTDIR}/ckpt/g-both-angle-s{s}.pt' for s in SEEDS}
for s in SEEDS:
    train('g-both-angle', s, save=CKPT[s], tag='readout')

In [ ]:
import torch
from grassman import data, readout
dev = torch.device('cuda')
val_x, val_y = data.blocks(data.load(DATA)['validation'], 128)
models = {s: readout.load(CKPT[s], dev) for s in SEEDS}
for s, m in models.items():
    print(f'seed {s}: validation perplexity of the saved weights {torch.exp(torch.tensor(readout.mean_nll(m, val_x, val_y))):.1f}')

## 1. Readable: edge use per layer

`share` is the average weight on each offset (positions where all six edges exist). `top` is the average weight of each token's strongest edge: 1/6 ≈ 0.17 would be no preference, 1.0 always one edge.

In [ ]:
import numpy as np
usage = {s: readout.edge_usage(m, val_x) for s, m in models.items()}
deltas = usage[SEEDS[0]][0]['deltas']
print('layer  seed  ' + ''.join(f't-{d:<6}' for d in deltas) + '  top')
for layer in usage[SEEDS[0]]:
    for s in SEEDS:
        u = usage[s][layer]
        print(f'{layer:>5}  {s:>4}  ' + ''.join(f'{x:<8.2f}' for x in u['share']) + f"  {u['top_weight']:.2f}")
same = [np.corrcoef(usage[SEEDS[0]][l]['share'], usage[SEEDS[1]][l]['share'])[0, 1] for l in usage[SEEDS[0]]]
print('\nagreement between seeds per layer (correlation of shares):', ' '.join(f'{c:.2f}' for c in same))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(SEEDS), figsize=(5 * len(SEEDS), 3.5))
for ax, s in zip(np.atleast_1d(axes), SEEDS):
    grid = np.array([usage[s][l]['share'] for l in usage[s]])
    im = ax.imshow(grid, cmap='viridis', vmin=0, vmax=grid.max(), aspect='auto')
    ax.set_xticks(range(len(deltas)), [f't-{d}' for d in deltas]); ax.set_yticks(range(len(grid)), [f'layer {l}' for l in usage[s]])
    ax.set_title(f'edge share, seed {s}')
    for (i, j), v in np.ndenumerate(grid):
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='w' if v < grid.max() / 2 else 'k', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUTDIR}/readout_edge_share.png', dpi=150); plt.show()

## 2. Used: remove one edge at a time

Increase in validation loss (nats per token) on the first 300 validation blocks. Compare with the shares above: an edge with a large share and a large increase is genuinely used.

In [ ]:
abl = {s: readout.edge_ablation(m, val_x[:300], val_y[:300]) for s, m in models.items()}
for s in SEEDS:
    print(f"seed {s}: base perplexity {abl[s]['base_ppl']:.1f}")
    print('layer  ' + ''.join(f't-{d:<7}' for d in deltas))
    for layer, row in abl[s]['delta_nll'].items():
        print(f'{layer:>5}  ' + ''.join(f'{row[d]:<+9.3f}' for d in deltas))

## 3. Meaningful: does sin(angle) mark sentence boundaries?

Mean sin(angle) between z_t and z_{t−1} when token t−1 is `.`, `!` or `?`, against everywhere else. The learned `angle_weight` shows whether each layer's edge scores use the angle at all.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
ends = tok.convert_tokens_to_ids(['.', '!', '?'])
for s, m in models.items():
    sb = readout.sin_by_boundary(m, val_x, ends)
    print(f'seed {s}')
    for layer, v in sb.items():
        w = readout.pair_layers(m)[layer][1].angle_weight.item()
        print(f"  layer {layer}: after a sentence end {v['after_boundary']:.3f}   elsewhere {v['elsewhere']:.3f}   "
              f"(n={v['n_boundary']:,}, angle weight {w:+.2f})")

## How to read it

- **Readable:** a high `top` value (well above 0.17) and the same pattern in both seeds mean each layer has a stable, nameable job ("layer 0 reads the previous token").
- **Used:** the edges with the biggest shares should also cost the most when removed. If removing them changes nothing, the readout is decoration.
- **Meaningful:** sin(angle) clearly higher after sentence ends means the angle is a 'something changed here' signal. If the angle weights stay near 0, the model ignores it.

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')